# Employee Workload & Attrition Analysis

### Classification Model to Predict Employee Attrition

## 1. Problem Statement

The goal of this project is to build a **classification model** that predicts whether an employee will leave the company (`attrition = 1`) or stay (`attrition = 0`) based on workload, performance, and job-related factors.

Dataset:
**Employee Workload and Attrition Analysis** (Kaggle)

## 2. Import Libraries

In [ ]:
# Basic Libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Model Selection
from sklearn.model_selection import train_test_split, cross_val_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    RocCurveDisplay
)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

## 3. Load Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/employee-workload-and-attrition-analysis/employee_performance_workload_attrition.csv')

df.head()

## 4. Basic Data Inspection

In [ ]:
print("Shape of dataset:", df.shape)
print("\nData Info:\n")
print(df.info())

print("\nMissing Values:\n")
print(df.isnull().sum())

**No missing values.**

## 5. Target Variable Encoding

Convert `attrition` into binary:

In [ ]:
df['attrition'] = (df['attrition'] == 'Yes').astype(int)
df['attrition'].value_counts()

## 6. Exploratory Data Analysis (EDA)

### 6.1 Attrition Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='attrition', data=df)
plt.title("Attrition Distribution")
plt.xticks([0,1], ['No', 'Yes'])
plt.show()

### 6.2 Correlation Heatmap (Numerical Features)

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(df.drop(['employee_id', 'department', 'role_level'], axis=1).corr(), 
            annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

### 6.3 Attrition vs Job Satisfaction

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='attrition', y='job_satisfaction', data=df)
plt.title("Attrition vs Job Satisfaction")
plt.show()

### 6.4 Attrition vs Avg Weekly Hours

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='attrition', y='avg_weekly_hours', data=df)
plt.title("Attrition vs Avg Weekly Hours")
plt.show()

# 7. Feature Engineering & Preprocessing

### Drop Unnecessary Columns

* `employee_id` → unique identifier → no predictive power

In [ ]:
df = df.drop('employee_id', axis=1)

## 8. Define Features & Target

In [ ]:
X = df.drop('attrition', axis=1)
y = df['attrition']

## 9. Identify Feature Types

In [ ]:
categorical_features = ['department']
ordinal_features = ['role_level']
numerical_features = [
    'monthly_salary',
    'avg_weekly_hours',
    'projects_handled',
    'performance_rating',
    'absences_days',
    'job_satisfaction'
]

## 10. Encoding Strategy

* **department → OneHotEncoder**
* **role_level → OrdinalEncoder (Junior < Mid < Senior)**
* **Numerical → StandardScaler**

In [ ]:
ordinal_categories = [['Junior', 'Mid', 'Senior']]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features),
        ('ord', OrdinalEncoder(categories=ordinal_categories), ordinal_features)
    ]
)

# 11. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# 12. Model Building

We will compare:

* Logistic Regression
* Random Forest
* Gradient Boosting

## 12.1 Logistic Regression Pipeline

In [ ]:
log_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

## 12.2 Random Forest

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

## 12.3 Gradient Boosting

In [ ]:
gb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

# 13. Model Evaluation Function

In [ ]:
def evaluate_model(name, y_test, y_pred, model, X_test):
    print(f"\n{name} Results")
    print("-"*40)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))
    
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

## 14. Compare Models

In [ ]:
evaluate_model("Logistic Regression", y_test, y_pred_log, log_model, X_test)
evaluate_model("Random Forest", y_test, y_pred_rf, rf_model, X_test)
evaluate_model("Gradient Boosting", y_test, y_pred_gb, gb_model, X_test)

# 15. ROC Curve Comparison

In [ ]:
plt.figure(figsize=(8,6))

for model, name in zip(
    [log_model, rf_model, gb_model],
    ["Logistic Regression", "Random Forest", "Gradient Boosting"]
):
    RocCurveDisplay.from_estimator(model, X_test, y_test)

plt.title("ROC Curve Comparison")
plt.show()

# 16. Feature Importance (Random Forest)

In [ ]:
# Extract feature names
ohe_features = rf_model.named_steps['preprocessor'] \
                        .named_transformers_['cat'] \
                        .get_feature_names_out(categorical_features)

all_features = numerical_features + list(ohe_features) + ordinal_features

importances = rf_model.named_steps['classifier'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10))
plt.title("Top 10 Important Features (Random Forest)")
plt.show()

# 17. Final Conclusion

### Key Findings:

* **Job Satisfaction** negatively correlates with attrition.
* **Absences Days** and **Weekly Hours** positively influence attrition.
* Tree-based models typically outperform linear models.
* Random Forest / Gradient Boosting generally provide better ROC-AUC.